# Speech Delivery Feedback Pipeline

Takes a recorded speech (audio file) and produces Toastmasters-style feedback on **pace, pitch/tone variation, loudness and pauses** -- comparing each sentence to (a) the speaker's overall average, (b) a rolling local baseline, and (c) its immediate neighbours (previous/next sentence).

Pipeline:
1. Transcribe with word + segment timestamps (Groq Whisper API)
2. Segment the transcript into sentence-level units
3. Extract prosodic features per unit: pitch (F0), loudness, speaking rate
4. Compute global baseline, rolling baseline, and local (neighbour) deltas
5. Apply perceptual thresholds to turn numbers into labeled events
6. Detect patterns across units (monotone stretches, sustained fast/slow runs)
7. Feed the structured events + transcript to a Groq-hosted open model to write rubric-style feedback

No audio-input LLM is used anywhere -- all acoustic analysis is classical DSP, and the LLM only ever sees text (transcript + computed numbers).

## Setup

```bash
pip install groq librosa soundfile numpy scipy praat-parselmouth
```

`praat-parselmouth` is optional but recommended for pitch tracking (wraps Praat). If it isn't installed, the notebook falls back to `librosa.pyin`.

Set your Groq API key as an environment variable before running. Groq provides hosted open models with a free tier, subject to current rate limits and model availability:
```bash
export GROQ_API_KEY=...
```

In [ ]:
import os
import re
import json
import math
from dataclasses import dataclass, field, asdict
from typing import List, Optional

import numpy as np
import librosa

try:
    import parselmouth
    HAVE_PARSELMOUTH = True
except ImportError:
    HAVE_PARSELMOUTH = False
    print('parselmouth not installed -- falling back to librosa.pyin for pitch tracking')

from groq import Groq

GROQ_API_KEY = os.environ.get('GROQ_API_KEY')

# Groq-hosted models are available through the free tier, subject to Groq's
# current rate limits and model availability.
GROQ_TRANSCRIPTION_MODEL = 'whisper-large-v3'
GROQ_FEEDBACK_MODEL = 'llama-3.3-70b-versatile'

groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

# --- Thresholds: a change below these is treated as 'not perceptually meaningful' ---
# Tune these against your own Toastmasters judgements once you have real recordings.
PACE_REL_THRESHOLD = 0.20       # 20% relative change in syllables/sec
PITCH_SEMITONE_THRESHOLD = 1.5  # semitones
LOUDNESS_DB_THRESHOLD = 2.5     # dB
MONOTONE_RUN_LENGTH = 4         # consecutive 'no-change' units to flag as a monotone stretch
SUSTAINED_RUN_LENGTH = 3        # consecutive same-direction units to flag as a sustained fast/slow run
ROLLING_WINDOW = 3              # units, for the rolling local baseline

## Step 1 -- Transcription (Groq Whisper, word + segment timestamps)

In [ ]:
def transcribe(audio_path: str) -> dict:
    """Transcribe audio_path with Groq Whisper, returning word- and segment-level timestamps.

    Groq's verbose_json response supports timestamp_granularities=['word','segment'] in the
    same call -- 'segment' gives full metadata (incl. avg_logprob, no_speech_prob), 'word'
    gives only {word, start, end}. We ask for both.
    """
    if groq_client is None:
        raise RuntimeError('Set GROQ_API_KEY before calling transcribe().')

    with open(audio_path, 'rb') as f:
        response = groq_client.audio.transcriptions.create(
            file=(os.path.basename(audio_path), f.read()),
            model=GROQ_TRANSCRIPTION_MODEL,
            response_format='verbose_json',
            timestamp_granularities=['word', 'segment'],
        )

    data = response.model_dump() if hasattr(response, 'model_dump') else json.loads(response.to_json())
    return data

# Example:
# transcript = transcribe('my_speech.wav')
# words = transcript['words']        # [{'word': 'Hello', 'start': 0.12, 'end': 0.34}, ...]
# segments = transcript['segments']  # whisper's own (coarser) segmentation, kept for reference

## Step 2 -- Segment into sentence-level units

Whisper's own `segments` can span more than one sentence, so we rebuild sentence boundaries
from the **word-level** timestamps and punctuation, which gives tighter control over what one
"unit" of comparison means.

In [ ]:
@dataclass
class Unit:
    index: int
    text: str
    start: float
    end: float
    words: List[dict]
    pause_before: Optional[float] = None
    pause_after: Optional[float] = None

SENTENCE_END_RE = re.compile(r'[.!?]+$')

def build_sentence_units(words: List[dict], max_gap_for_boundary: float = 0.0) -> List[Unit]:
    """Group words into sentence-level units using terminal punctuation.

    If Whisper produces a long run with no terminal punctuation (common with filler-heavy or
    run-on speech), you may want to also force a boundary after N seconds -- not done here to
    keep this a straightforward starting point.
    """
    units = []
    current = []
    for w in words:
        current.append(w)
        token = w['word'].strip()
        if SENTENCE_END_RE.search(token):
            units.append(current)
            current = []
    if current:
        units.append(current)

    result = []
    for i, u in enumerate(units):
        text = ' '.join(w['word'].strip() for w in u)
        start, end = u[0]['start'], u[-1]['end']
        result.append(Unit(index=i, text=text, start=start, end=end, words=u))

    # pause_before / pause_after from the gap to neighbouring units.
    # NOTE: word timestamps from Whisper can drift ~100-200ms. If you've already built a VAD
    # (see your own pause-detection project), swap these gaps for VAD-derived silence regions --
    # it will be more reliable than relying on Whisper's word boundaries alone.
    for i, u in enumerate(result):
        u.pause_before = u.start - result[i - 1].end if i > 0 else None
        u.pause_after = result[i + 1].start - u.end if i < len(result) - 1 else None

    return result

## Step 3 -- Prosodic feature extraction per unit

Pitch (F0) and loudness are computed once over the whole audio file, then sliced per unit by
timestamp -- this is both faster and avoids edge artifacts from analyzing tiny clips.

In [ ]:
def extract_f0_track(y: np.ndarray, sr: int, fmin: float = 75.0, fmax: float = 400.0):
    """Return (times, f0) with unvoiced frames as nan. fmin/fmax cover typical adult speech;
    narrow them if you know the speaker's gender/range, to cut octave-jump errors."""
    if HAVE_PARSELMOUTH:
        snd = parselmouth.Sound(y, sampling_frequency=sr)
        pitch = snd.to_pitch(time_step=0.01, pitch_floor=fmin, pitch_ceiling=fmax)
        times = pitch.ts()
        f0 = pitch.selected_array['frequency']
        f0 = np.where(f0 == 0, np.nan, f0)
        return times, f0
    else:
        f0, voiced_flag, _ = librosa.pyin(y, fmin=fmin, fmax=fmax, sr=sr)
        times = librosa.times_like(f0, sr=sr)
        f0 = np.where(voiced_flag, f0, np.nan)
        return times, f0

def extract_loudness_track(y: np.ndarray, sr: int, frame_length: int = 2048, hop_length: int = 512):
    """Return (times, db) -- RMS energy converted to dB (relative, not calibrated SPL)."""
    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    db = librosa.amplitude_to_db(rms, ref=1e-5)
    times = librosa.times_like(rms, sr=sr, hop_length=hop_length)
    return times, db

def slice_track(times: np.ndarray, values: np.ndarray, t0: float, t1: float) -> np.ndarray:
    mask = (times >= t0) & (times <= t1)
    return values[mask]

## Step 4 -- Speaking rate (syllables/sec)

In [ ]:
VOWEL_GROUP_RE = re.compile(r'[aeiouy]+', re.IGNORECASE)

def estimate_syllables(word: str) -> int:
    """Cheap heuristic syllable counter (vowel-group counting with common English adjustments).
    Good enough for a relative pace measure; not linguistically exact. Swap in `pyphen` or
    `syllapy` later if you want more accuracy."""
    w = re.sub(r"[^a-zA-Z']", '', word).lower()
    if not w:
        return 0
    groups = VOWEL_GROUP_RE.findall(w)
    count = len(groups)
    if w.endswith('e') and not w.endswith('le') and count > 1:
        count -= 1
    return max(count, 1)

def unit_speaking_rate(unit: Unit) -> dict:
    """Syllables/sec using only time actually spent speaking inside the unit (sum of each
    word's own duration), not wall-clock duration -- so internal micro-pauses don't get
    counted as 'speaking slowly'."""
    syllables = sum(estimate_syllables(w['word']) for w in unit.words)
    speaking_time = sum(max(w['end'] - w['start'], 0.0) for w in unit.words)
    speaking_time = max(speaking_time, 1e-6)
    return {
        'syllables': syllables,
        'speaking_time': speaking_time,
        'syll_per_sec': syllables / speaking_time,
        'wall_duration': unit.end - unit.start,
    }

## Step 5 -- Per-unit feature table

Pulls pitch, loudness and pace together into one record per sentence.

In [ ]:
@dataclass
class UnitFeatures:
    index: int
    text: str
    start: float
    end: float
    pause_before: Optional[float]
    pause_after: Optional[float]
    syll_per_sec: float
    pitch_median_hz: float
    pitch_p10_hz: float
    pitch_p90_hz: float
    loudness_mean_db: float
    loudness_peak_db: float

def compute_unit_features(units: List[Unit], y: np.ndarray, sr: int) -> List[UnitFeatures]:
    f0_times, f0_vals = extract_f0_track(y, sr)
    db_times, db_vals = extract_loudness_track(y, sr)

    features = []
    for u in units:
        rate = unit_speaking_rate(u)

        f0_slice = slice_track(f0_times, f0_vals, u.start, u.end)
        f0_voiced = f0_slice[~np.isnan(f0_slice)]
        if len(f0_voiced) == 0:
            pitch_med, pitch_p10, pitch_p90 = float('nan'), float('nan'), float('nan')
        else:
            pitch_med = float(np.median(f0_voiced))
            pitch_p10 = float(np.percentile(f0_voiced, 10))
            pitch_p90 = float(np.percentile(f0_voiced, 90))

        db_slice = slice_track(db_times, db_vals, u.start, u.end)
        loud_mean = float(np.mean(db_slice)) if len(db_slice) else float('nan')
        loud_peak = float(np.max(db_slice)) if len(db_slice) else float('nan')

        features.append(UnitFeatures(
            index=u.index, text=u.text, start=u.start, end=u.end,
            pause_before=u.pause_before, pause_after=u.pause_after,
            syll_per_sec=rate['syll_per_sec'],
            pitch_median_hz=pitch_med, pitch_p10_hz=pitch_p10, pitch_p90_hz=pitch_p90,
            loudness_mean_db=loud_mean, loudness_peak_db=loud_peak,
        ))
    return features

## Step 6 -- Baselines: global, rolling, and neighbour deltas

- **Global**: compare each unit to the speaker's own median across the whole speech.
- **Rolling**: compare to the median of the last `ROLLING_WINDOW` units (did they settle into one gear?).
- **Local**: compare to the immediately preceding and following unit (did *this* sentence stand out?).

Pitch is converted to semitones relative to the speaker's own global median F0, so results are
comparable across voices/genders and not just raw Hz.

In [ ]:
def hz_to_semitones(hz: float, ref_hz: float) -> float:
    if hz is None or ref_hz is None or math.isnan(hz) or math.isnan(ref_hz) or hz <= 0 or ref_hz <= 0:
        return float('nan')
    return 12.0 * math.log2(hz / ref_hz)

def compute_baselines(features: List[UnitFeatures]) -> dict:
    pace = np.array([f.syll_per_sec for f in features])
    pitch_hz = np.array([f.pitch_median_hz for f in features])
    loud = np.array([f.loudness_mean_db for f in features])

    global_pace = float(np.nanmedian(pace))
    global_pitch_hz = float(np.nanmedian(pitch_hz))
    global_loud = float(np.nanmean(loud))

    return {'global_pace': global_pace, 'global_pitch_hz': global_pitch_hz, 'global_loud': global_loud}

def rolling_median(values: List[float], i: int, window: int) -> float:
    lo = max(0, i - window)
    window_vals = [v for v in values[lo:i] if v is not None and not (isinstance(v, float) and math.isnan(v))]
    if not window_vals:
        return float('nan')
    return float(np.median(window_vals))

def pct_change(new, old):
    if old in (0, None) or (isinstance(old, float) and math.isnan(old)):
        return float('nan')
    if new is None or (isinstance(new, float) and math.isnan(new)):
        return float('nan')
    return (new - old) / abs(old)

## Step 7 -- Turn numbers into labeled events

In [ ]:
def label_change(value: float, threshold: float, higher_label: str, lower_label: str, none_label: str = 'steady') -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return none_label
    if value > threshold:
        return higher_label
    if value < -threshold:
        return lower_label
    return none_label

def build_unit_events(features: List[UnitFeatures], baselines: dict) -> List[dict]:
    ref_pitch_hz = baselines['global_pitch_hz']
    pace_list = [f.syll_per_sec for f in features]
    pitch_list = [f.pitch_median_hz for f in features]
    loud_list = [f.loudness_mean_db for f in features]

    events = []
    for i, f in enumerate(features):
        pitch_st = hz_to_semitones(f.pitch_median_hz, ref_pitch_hz)
        pitch_range_st = (
            hz_to_semitones(f.pitch_p90_hz, f.pitch_p10_hz) if not math.isnan(f.pitch_p10_hz) else float('nan')
        )

        # vs global
        pace_vs_global = pct_change(f.syll_per_sec, baselines['global_pace'])
        loud_vs_global = f.loudness_mean_db - baselines['global_loud'] if not math.isnan(f.loudness_mean_db) else float('nan')

        # vs rolling local baseline (previous ROLLING_WINDOW units)
        pace_roll = rolling_median(pace_list, i, ROLLING_WINDOW)
        loud_roll = rolling_median(loud_list, i, ROLLING_WINDOW)
        pace_vs_roll = pct_change(f.syll_per_sec, pace_roll)
        loud_vs_roll = f.loudness_mean_db - loud_roll if not math.isnan(loud_roll) else float('nan')

        # vs previous / next unit (neighbour contrast)
        pace_vs_prev = pct_change(f.syll_per_sec, pace_list[i - 1]) if i > 0 else float('nan')
        pace_vs_next = pct_change(f.syll_per_sec, pace_list[i + 1]) if i < len(features) - 1 else float('nan')
        pitch_vs_prev = hz_to_semitones(f.pitch_median_hz, pitch_list[i - 1]) if i > 0 else float('nan')
        pitch_vs_next = hz_to_semitones(f.pitch_median_hz, pitch_list[i + 1]) if i < len(features) - 1 else float('nan')
        loud_vs_prev = f.loudness_mean_db - loud_list[i - 1] if i > 0 else float('nan')
        loud_vs_next = f.loudness_mean_db - loud_list[i + 1] if i < len(features) - 1 else float('nan')

        events.append({
            'index': i,
            'text': f.text,
            'start': round(f.start, 2),
            'end': round(f.end, 2),
            'pause_before_s': round(f.pause_before, 2) if f.pause_before is not None else None,
            'pause_after_s': round(f.pause_after, 2) if f.pause_after is not None else None,
            'pace_syll_per_sec': round(f.syll_per_sec, 2),
            'pace_label_vs_prev': label_change(pace_vs_prev, PACE_REL_THRESHOLD, 'faster', 'slower'),
            'pace_label_vs_next': label_change(pace_vs_next, PACE_REL_THRESHOLD, 'faster', 'slower'),
            'pace_label_vs_global': label_change(pace_vs_global, PACE_REL_THRESHOLD, 'faster', 'slower'),
            'pitch_semitone_vs_speaker_median': round(pitch_st, 2) if not math.isnan(pitch_st) else None,
            'pitch_range_semitones': round(pitch_range_st, 2) if not math.isnan(pitch_range_st) else None,
            'pitch_label_vs_prev': label_change(pitch_vs_prev, PITCH_SEMITONE_THRESHOLD, 'higher', 'lower'),
            'pitch_label_vs_next': label_change(pitch_vs_next, PITCH_SEMITONE_THRESHOLD, 'higher', 'lower'),
            'loudness_db_vs_global': round(loud_vs_global, 2) if not math.isnan(loud_vs_global) else None,
            'loudness_label_vs_prev': label_change(loud_vs_prev, LOUDNESS_DB_THRESHOLD, 'louder', 'softer'),
            'loudness_label_vs_next': label_change(loud_vs_next, LOUDNESS_DB_THRESHOLD, 'louder', 'softer'),
            'loudness_label_vs_rolling': label_change(loud_vs_roll, LOUDNESS_DB_THRESHOLD, 'louder', 'softer'),
            'pace_label_vs_rolling': label_change(pace_vs_roll, PACE_REL_THRESHOLD, 'faster', 'slower'),
        })
    return events

## Step 8 -- Pattern detection across units

Single-unit labels tell you about one sentence. Patterns tell you about the *shape* of the
whole delivery -- e.g. a monotone stretch, or a run that keeps accelerating.

In [ ]:
def detect_patterns(events: List[dict]) -> List[dict]:
    patterns = []

    # Monotone run: consecutive units where pace, pitch AND loudness are all 'steady' vs rolling/prev
    run_start = None
    for i, e in enumerate(events):
        is_flat = (
            e['pace_label_vs_rolling'] == 'steady'
            and e['pitch_label_vs_prev'] == 'steady'
            and e['loudness_label_vs_rolling'] == 'steady'
        )
        if is_flat:
            if run_start is None:
                run_start = i
        else:
            if run_start is not None and i - run_start >= MONOTONE_RUN_LENGTH:
                patterns.append({
                    'type': 'monotone_stretch',
                    'start_index': run_start, 'end_index': i - 1,
                    'start_time': events[run_start]['start'], 'end_time': events[i - 1]['end'],
                })
            run_start = None
    if run_start is not None and len(events) - run_start >= MONOTONE_RUN_LENGTH:
        patterns.append({
            'type': 'monotone_stretch',
            'start_index': run_start, 'end_index': len(events) - 1,
            'start_time': events[run_start]['start'], 'end_time': events[-1]['end'],
        })

    # Sustained fast/slow run vs the speaker's own global pace
    for direction in ('faster', 'slower'):
        run_start = None
        for i, e in enumerate(events):
            if e['pace_label_vs_global'] == direction:
                if run_start is None:
                    run_start = i
            else:
                if run_start is not None and i - run_start >= SUSTAINED_RUN_LENGTH:
                    patterns.append({
                        'type': f'sustained_{direction}',
                        'start_index': run_start, 'end_index': i - 1,
                        'start_time': events[run_start]['start'], 'end_time': events[i - 1]['end'],
                    })
                run_start = None
        if run_start is not None and len(events) - run_start >= SUSTAINED_RUN_LENGTH:
            patterns.append({
                'type': f'sustained_{direction}',
                'start_index': run_start, 'end_index': len(events) - 1,
                'start_time': events[run_start]['start'], 'end_time': events[-1]['end'],
            })

    # No pause before a unit that is both louder AND higher-pitched than its neighbours
    # (a likely 'key point' that wasn't set up with a pause)
    for e in events:
        if (e['pause_before_s'] is not None and e['pause_before_s'] < 0.15
                and e['loudness_label_vs_prev'] == 'louder'
                and e['pitch_label_vs_prev'] == 'higher'):
            patterns.append({
                'type': 'emphasis_without_pause',
                'start_index': e['index'], 'end_index': e['index'],
                'start_time': e['start'], 'end_time': e['end'],
            })

    return patterns

## Step 9 -- Feedback synthesis (text-only LLM)

The LLM never touches audio or raw numbers -- it gets the transcript plus already-labeled
events and patterns, and its only job is to turn them into specific, rubric-style coaching
notes. This keeps the feedback grounded in what was actually measured.

In [ ]:
FEEDBACK_SYSTEM_PROMPT = """You are an experienced Toastmasters evaluator giving verbal delivery feedback \
 on a speech. You will be given the transcript split into sentences, and for each sentence a set \
 of pre-computed, already-labeled measurements about pace, pitch and loudness, each compared to \
 the previous sentence, the next sentence, and the speaker's own overall average. You will also \
 get a list of detected patterns (e.g. monotone stretches, sustained fast/slow runs).

Rules:
- Only comment on things supported by the data you were given. Do not invent measurements.
- Give 2-4 specific strengths and 2-4 specific areas to improve, each tied to an exact sentence \
 or time range from the data.
- Prefer concrete, actionable notes ('slow down on your key line at 0:42') over vague praise.
- Keep it in the voice of a supportive but direct Toastmasters evaluator.
"""

def generate_feedback(events: List[dict], patterns: List[dict], model: str = GROQ_FEEDBACK_MODEL) -> str:
    if groq_client is None:
        raise RuntimeError('Set GROQ_API_KEY before calling generate_feedback().')

    payload = {'sentences': events, 'patterns': patterns}
    user_prompt = (
        'Here is the measured delivery data for one speech, as JSON:\n\n'
        + json.dumps(payload, indent=2)
        + '\n\nWrite the evaluator feedback now.'
    )

    response = groq_client.chat.completions.create(
        model=model,
        max_tokens=1500,
        temperature=0.2,
        messages=[
            {'role': 'system', 'content': FEEDBACK_SYSTEM_PROMPT},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content or ''

## Step 10 -- End-to-end pipeline

In [ ]:
def analyze_speech(audio_path: str) -> dict:
    y, sr = librosa.load(audio_path, sr=None, mono=True)

    transcript = transcribe(audio_path)
    units = build_sentence_units(transcript['words'])
    features = compute_unit_features(units, y, sr)
    baselines = compute_baselines(features)
    events = build_unit_events(features, baselines)
    patterns = detect_patterns(events)
    feedback = generate_feedback(events, patterns)

    return {
        'baselines': baselines,
        'events': events,
        'patterns': patterns,
        'feedback': feedback,
    }

# Example run (needs a real audio file + both API keys set):
# result = analyze_speech('my_speech.wav')
# print(result['feedback'])
# result['events']    # per-sentence table, useful for a timeline UI later
# result['patterns']  # flagged stretches, useful for highlighting on that timeline

## Notes / next steps

- **Thresholds** (`PACE_REL_THRESHOLD`, `PITCH_SEMITONE_THRESHOLD`, `LOUDNESS_DB_THRESHOLD`, run 
  lengths) are starting guesses. Record a few real speeches, write your own evaluation of each, 
  and tune these numbers until the flagged events match your own ear.
- **Pauses**: this notebook approximates pauses from the gaps between Whisper's word 
  timestamps, which can drift ~100-200ms. Swap in your own VAD project's silence regions for 
  `pause_before` / `pause_after` -- it will be materially more accurate than word-gap timing.
- **Sentence segmentation** here is pure punctuation-splitting. Long run-on or filler-heavy 
  speech may produce very long 'sentences' -- consider also forcing a split after a fixed max 
  duration, or after any pause longer than ~1.5s.
- **Content-aware layer (not yet included)**: tagging each sentence's rhetorical role (hook, 
  story, key point, closing line) with a text LLM, then comparing *delivered* vs *needed* 
  emphasis, is the next layer of depth once the numeric pipeline above is validated.
- **Emotion/intimacy detection** is deliberately left out -- it's the least reliable piece and 
  worth tackling last, once pace/pitch/loudness feedback is already solid.